# Cross-Encoder Reranker — 2단계 검색 품질 향상

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Bi-Encoder**와 **Cross-Encoder**의 차이 및 각각의 역할 이해하기
- HuggingFace의 `BAAI/bge-reranker-v2-m3` 모델로 재순위화(Reranking) 구현하기
- `ContextualCompressionRetriever`를 Reranker 파이프라인에 통합하기

## 사전 지식

- VectorStoreRetriever와 임베딩 기반 검색의 작동 방식
- Bi-Encoder: 쿼리와 문서를 각각 독립적으로 인코딩하는 방식 (빠르지만 상호작용 없음)

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> BE[Bi-Encoder<br/>벡터 검색<br/>1차: k=10]:::process
    BE --> RE[Cross-Encoder<br/>Reranker<br/>2차: top_n=3]:::process
    RE --> R[정밀 선별된<br/>상위 3개 문서]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## Two-Stage Retrieval

| 단계 | 방식 | 역할 | 속도 |
|------|------|------|------|
| **1단계** | Bi-Encoder (벡터 검색) | 후보 문서 빠르게 추출 (k=10~50) | 빠름 |
| **2단계** | Cross-Encoder (Reranker) | 쿼리-문서 상호작용 분석, 정밀 재정렬 | 느리지만 정확 |

Cross-Encoder는 쿼리와 문서를 **쌍(pair)**으로 함께 분석해서 더 정확한 관련성을 계산해요.

> **주의**: Cross-Encoder 모델(특히 BAAI/bge-reranker-v2-m3)은 처음 실행 시 HuggingFace에서 모델을 다운로드하므로 시간이 걸릴 수 있어요.

## 1. 환경 설정

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 필요한 라이브러리 import
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# 환경 변수 로드
load_dotenv()

In [ ]:
# 문서 출력 도우미 함수
def pretty_print_docs(docs, show_metadata=False):
    """검색된 문서를 보기 좋게 출력하는 함수"""
    print(f"\n{'=' * 100}")
    print(f"총 {len(docs)}개 문서 검색됨")
    print(f"{'=' * 100}\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        print(f"내용: {doc.page_content}")
        
        if show_metadata and doc.metadata:
            # relevance_score가 있으면 표시 (Reranker 결과인 경우)
            if 'relevance_score' in doc.metadata:
                print(f"관련성 점수: {doc.metadata['relevance_score']:.4f}")
            print(f"메타데이터: {doc.metadata}")
        
        print(f"{'─' * 100}\n")


## 2. 문서 준비

### 2.1 문서 로드 및 분할

In [ ]:
# 1단계: 문서 로드

# 아래에 코드를 작성하세요


In [ ]:
# 2단계: 텍스트 분할

# 아래에 코드를 작성하세요


### 2.2 벡터스토어 생성 및 기본 검색

In [ ]:
# ---------------------------------------------------
# HuggingFace 임베딩으로 FAISS 벡터스토어 생성 — Reranker 비교 기준선 확립
# ---------------------------------------------------

# ============================================================
# TODO: HuggingFaceEmbeddings로 임베딩하고 FAISS 벡터스토어를 만드세요
# 힌트: k=10으로 설정해야 Reranker가 충분한 후보 문서를 받을 수 있음
# 예상 결과: 벡터스토어 생성 완료 메시지 출력
# ============================================================

# 임베딩 모델 설정 (한국어/영어 모두 지원하는 모델)
# paraphrase-multilingual-mpnet-base-v2: 다국어 지원 Sentence-BERT 계열

# FAISS 벡터스토어 생성 및 검색기 설정


In [ ]:
# ---------------------------------------------------
# Reranker 적용 전 기본 벡터 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: base_retriever로 쿼리를 검색하고 결과를 출력하세요
# 힌트: invoke(query) 호출 후 pretty_print_docs()로 출력
# 예상 결과: 10개 문서가 유사도 순으로 출력됨
# ============================================================

# 기본 검색 테스트


# Reranker 없이 기본 벡터 검색 수행


## 3. Cross-Encoder Reranker 적용

이제 Cross-Encoder 모델을 사용하여 검색 결과를 재정렬합니다.

### 3.1 Cross-Encoder 모델 선택

다양한 Cross-Encoder 모델이 있습니다:

- `BAAI/bge-reranker-v2-m3`: 다국어 지원, 높은 정확도 (권장)
- `cross-encoder/ms-marco-MiniLM-L-6-v2`: 영어 특화, 빠른 속도
- `cross-encoder/ms-marco-MiniLM-L-12-v2`: 영어 특화, 높은 정확도

In [ ]:
# 1단계: Cross-Encoder 모델 초기화
# BAAI/bge-reranker-v2-m3: 한국어/영어 등 다국어를 지원하는 고성능 Reranker

# 2단계: Reranker 설정 (상위 3개만 선택)
# top_n=3: 10개 후보 중 가장 관련성 높은 3개만 최종 반환

# 3단계: 압축 검색기 생성
# base_retriever → 10개 추출 → reranker → 상위 3개 선별
# ⚠️ 주의: ContextualCompressionRetriever를 재사용해 Reranker를 압축기로 등록

# 아래에 코드를 작성하세요


In [ ]:
# ---------------------------------------------------
# Reranker 적용 후 검색 결과 확인
# ---------------------------------------------------
# ============================================================
# TODO: compression_retriever로 같은 쿼리를 검색하고 결과를 비교하세요
# 힌트: show_metadata=True로 관련성 점수(relevance_score)를 함께 출력
# 예상 결과: 10개 → 3개로 줄어들고 관련성 점수가 표시됨
# ============================================================

# Reranker 적용 검색 테스트

# Reranker 적용 검색


## 4. 결과 비교 및 분석

Reranker 적용 전후를 비교하여 효과를 확인합니다.

In [ ]:
# ---------------------------------------------------
# Reranker 적용 전후 결과 비교 분석
# ---------------------------------------------------


    # 첫 100자만 표시

# 아래에 코드를 작성하세요


## 5. 다양한 쿼리로 테스트

여러 유형의 쿼리로 Reranker의 효과를 확인합니다.

In [ ]:
# ---------------------------------------------------
# 다양한 쿼리로 Reranker 성능 테스트
# ---------------------------------------------------
# ============================================================
# TODO: 여러 쿼리를 순회하며 각각 compression_retriever로 검색하세요
# 힌트: for 루프로 test_queries를 순회하며 invoke() 호출
# 예상 결과: 각 쿼리별 상위 3개 문서와 관련성 점수 출력
# ============================================================

# 테스트 쿼리 목록


    
    # Reranker 적용
    
    
        # 첫 80자만 표시


## 6. 핵심 정리

### Reranker의 주요 장점

1. **정확도 향상**
   - Cross-Encoder가 쿼리-문서 간 상호작용을 직접 분석
   - Bi-Encoder보다 훨씬 정확한 관련성 평가

2. **비용 효율성**
   - 초기 검색: 많은 후보를 빠르게 추출
   - 재정렬: 소수 문서만 정밀 분석
   - LLM 입력 토큰 수 절감

3. **유연한 통합**
   - 기존 검색 시스템에 쉽게 추가 가능
   - 다양한 Cross-Encoder 모델 선택 가능

### 사용 시 고려사항

- **초기 검색 수 (k)**: 충분한 후보 확보 (보통 10~50개)
- **최종 문서 수 (top_n)**: LLM 컨텍스트 크기 고려 (보통 3~5개)
- **모델 선택**: 
  - 다국어: `BAAI/bge-reranker-v2-m3`
  - 영어 특화: `cross-encoder/ms-marco-MiniLM-L-12-v2`
  - 속도 중시: 경량 모델 (L-6 버전)

### Two-Stage Retrieval 권장 설정

```python
# Stage 1: 벡터 검색 (빠르게 후보 추출)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# Stage 2: Cross-Encoder 재정렬 (정확하게 선별)
reranker = CrossEncoderReranker(model=cross_encoder_model, top_n=5)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)
```

> **실무 팁**: Cross-Encoder는 GPU가 있으면 훨씬 빠릅니다. CPU 환경에서 느리다면 더 경량 모델(L-6 버전)을 선택하거나, 클라우드 기반 Reranker(Cohere, Jina)로 전환하세요.

## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

### Two-Stage Retrieval 권장 설정

```python
# 1단계: 빠른 후보 추출
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# 2단계: 정밀 재정렬 (상위 5개만 최종 선택)
reranker = CrossEncoderReranker(model=cross_encoder_model, top_n=5)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)
```

### 다음 노트북 예고

**02-Cohere-Reranker**: 로컬 모델 대신 Cohere의 클라우드 API를 활용한 Reranker를 배워요. 인프라 관리 없이 프로덕션 수준의 재순위화를 구현합니다.